# 1.6 · 高级 SQL 题型 / LeetCode-Style SQL Patterns ⭐

> **课程定位 / Where this fits**
> **Part 1 第 6 课**——把 1.1-1.5 学的全套工具甩在大厂面试 SQL 题上。**这一节就是面试前夜的 cheat sheet**。
> **Part 1, lesson 6** — drill all the tools from 1.1-1.5 on the canonical big-tech SQL interview problems. **This lesson is the night-before-interview cheat sheet.**

> 💡 **覆盖的 10 个高频题型 / 10 high-frequency patterns covered**
> 1. N-th highest（第 N 高）
> 2. 中位数（不用 `MEDIAN` 函数）
> 3. "买过全部商品"的客户（**关系除法**）
> 4. Pivot 行转列（动态 + 静态）
> 5. 转化漏斗 / Funnel
> 6. **Sessionization**（会话切分，间隔 > X 切新会话）
> 7. **留存率 D1 / D7 / D30**
> 8. **Cohort 分析**
> 9. 重复检测 + 去重
> 10. 经理层级 / SELF JOIN 链
>
> 学完这一节，**LeetCode SQL "Hard" 大半秒杀**。

---

## 目录 / Table of Contents

1. [扩展数据集：events 表 / Extended Dataset](#0)
2. [Problem 1 · N-th 高值 / Nth Highest](#1)
3. [Problem 2 · 中位数（手算）/ Median Without MEDIAN](#2)
4. [Problem 3 · 买过"全部"商品（关系除法）/ Relational Division](#3)
5. [Problem 4 · Pivot 行转列 / Rows-to-columns](#4)
6. [Problem 5 · 转化漏斗 / Funnel](#5)
7. [Problem 6 · Sessionization](#6)
8. [Problem 7 · 留存率 D1 / D7 / D30](#7)
9. [Problem 8 · Cohort 分析](#8)
10. [Problem 9 · 找/删重复 / Find & Dedup Duplicates](#9)
11. [Problem 10 · 经理层级 / Manager Chain](#10)
12. [小结 / Summary](#11)


<a id="0"></a>
## 0. 扩展数据集 / Extended Dataset

继续用 Mini Music Store，但再加 3 张表：`events`（用户行为日志）、`employee`（员工层级）、`product`（用来演示"买过全部"）。
We extend the Mini Music Store with three more tables.

| 新表 | 用途 |
|---|---|
| `events`     | 用户行为日志（view/play/like/buy），用于 sessionization / 漏斗 / 留存 |
| `employee`   | 员工 + manager_id，用于 SELF JOIN 层级 |
| `product_required` | "必买"商品清单，用于"买过全部"的关系除法 |


In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

conn = duckdb.connect()

# 1) 重建 Mini Music Store / Rebuild original tables
conn.sql("""
CREATE TABLE artist (artist_id INT PRIMARY KEY, name VARCHAR, country VARCHAR);
INSERT INTO artist VALUES
    (1,'The Beatles','UK'), (2,'Pink Floyd','UK'), (3,'Miles Davis','US'),
    (4,'Daft Punk','FR'), (5,'Radiohead','UK'), (6,'Anonymous Artist',NULL);

CREATE TABLE customer (customer_id INT PRIMARY KEY, name VARCHAR, country VARCHAR, salary DECIMAL(10,2));
INSERT INTO customer VALUES
    (1,'Alice Chen','US',95000),
    (2,'Bob Smith','UK',75000),
    (3,'Charlie Davis','US',120000),
    (4,'Diana Park','DE',95000),     -- 故意和 Alice 同薪 / intentional tie
    (5,'Ethan Miller','US',85000),
    (6,'Fiona Wong','JP',60000),
    (7,'George Lee','US',120000),    -- 和 Charlie 同薪 / tie with Charlie
    (8,'Hannah Kim','UK',55000);
""")

# 2) Events 表 / Events table
conn.sql("""
CREATE TABLE events (
    event_id   INT PRIMARY KEY,
    user_id    INT,
    event_type VARCHAR,
    ts         TIMESTAMP
);
""")

# 制造 events：5 个用户、5 天、混合事件 / Synthetic event log
# 包含明显的会话切分（同一用户两次事件间隔 > 30 min 算新 session）
conn.sql("""
INSERT INTO events VALUES
    -- user 1: 1 天内 1 session (3 events) + 第二天 1 session
    (1, 1, 'view',  TIMESTAMP '2026-06-01 10:00:00'),
    (2, 1, 'play',  TIMESTAMP '2026-06-01 10:05:00'),
    (3, 1, 'buy',   TIMESTAMP '2026-06-01 10:15:00'),
    (4, 1, 'view',  TIMESTAMP '2026-06-02 09:00:00'),
    (5, 1, 'play',  TIMESTAMP '2026-06-02 09:02:00'),
    -- user 2: 单日多 session（中间间隔大）
    (6, 2, 'view',  TIMESTAMP '2026-06-01 11:00:00'),
    (7, 2, 'play',  TIMESTAMP '2026-06-01 11:05:00'),
    (8, 2, 'view',  TIMESTAMP '2026-06-01 14:30:00'),   -- gap > 30 min → new session
    (9, 2, 'buy',   TIMESTAMP '2026-06-01 14:40:00'),
    -- user 3: 单 event
    (10,3, 'view',  TIMESTAMP '2026-06-02 13:00:00'),
    -- user 4: 漏斗完整走完 / Full funnel
    (11,4, 'view',  TIMESTAMP '2026-06-03 08:00:00'),
    (12,4, 'play',  TIMESTAMP '2026-06-03 08:05:00'),
    (13,4, 'like',  TIMESTAMP '2026-06-03 08:07:00'),
    (14,4, 'buy',   TIMESTAMP '2026-06-03 08:20:00'),
    -- user 5: 多日活跃
    (15,5, 'view',  TIMESTAMP '2026-06-01 12:00:00'),
    (16,5, 'view',  TIMESTAMP '2026-06-02 12:00:00'),
    (17,5, 'play',  TIMESTAMP '2026-06-02 12:10:00'),
    (18,5, 'view',  TIMESTAMP '2026-06-04 09:00:00'),
    (19,5, 'play',  TIMESTAMP '2026-06-04 09:01:00'),
    (20,5, 'buy',   TIMESTAMP '2026-06-04 09:30:00');
""")

# 3) Employee 表 / Employees with manager hierarchy
conn.sql("""
CREATE TABLE employee (
    emp_id     INT PRIMARY KEY,
    name       VARCHAR,
    manager_id INT,                 -- NULL = CEO
    salary     DECIMAL(10,2)
);
INSERT INTO employee VALUES
    (1, 'CEO Carla',      NULL,  300000),
    (2, 'VP Vincent',     1,     200000),
    (3, 'VP Vera',        1,     200000),
    (4, 'Mgr Mark',       2,     130000),
    (5, 'Mgr Maya',       2,     135000),
    (6, 'Mgr Miguel',     3,     130000),
    (7, 'Eng Erik',       4,     90000),
    (8, 'Eng Emma',       4,     95000),
    (9, 'Eng Eli',        5,     95000),
    (10,'Eng Eva',        6,     90000);
""")

# 4) product_required: 必买商品清单 / Required-product list
conn.sql("""
CREATE TABLE product_required (product_id INT PRIMARY KEY, name VARCHAR);
INSERT INTO product_required VALUES
    (101,'phone'), (102,'charger'), (103,'case');

CREATE TABLE orders_demo (
    order_id   INT PRIMARY KEY,
    user_id    INT,
    product_id INT
);
INSERT INTO orders_demo VALUES
    -- user 1 买了全部 3 个 / bought all 3
    (1,1,101), (2,1,102), (3,1,103),
    -- user 2 只买了 2 个 / bought only 2
    (4,2,101), (5,2,102),
    -- user 3 也买全 / bought all 3
    (6,3,101), (7,3,102), (8,3,103),
    -- user 4 啥也没买（不在 orders 里）
    (9,5,103);
""")

print(f"duckdb : {duckdb.__version__}")
print(f"tables : {conn.sql('SHOW TABLES').df()['name'].tolist()}")


<a id="1"></a>
## Problem 1 · N-th 高值 / Nth Highest

> "**找出第 N 高的薪水**，相同薪水算并列。"
> "Find the 2nd highest salary; ties count as same rank."

**这是 LeetCode SQL 第 1 道经典**。陷阱在于"有并列"——用 `DENSE_RANK` 还是 `LIMIT` 答出不同结果。
LeetCode SQL Problem #1 — the catch is ties.

### 三种典型解法 / Three classic solutions


In [ ]:
# 数据预览：customer 表的工资 / Preview salaries
conn.sql("""
    SELECT name, salary
    FROM customer
    ORDER BY salary DESC;
""").df()


In [ ]:
# 解法 A: DENSE_RANK ⭐（推荐）/ Solution A: DENSE_RANK (recommended)
# 第 2 高 = DENSE_RANK = 2 的所有人
conn.sql("""
    WITH ranked AS (
        SELECT
            name,
            salary,
            DENSE_RANK() OVER (ORDER BY salary DESC) AS rk
        FROM customer
    )
    SELECT name, salary
    FROM ranked
    WHERE rk = 2;
""").df()


In [ ]:
# 解法 B: 子查询 + DISTINCT + LIMIT/OFFSET (老派但通用)
# Old-school: distinct + offset
conn.sql("""
    SELECT name, salary
    FROM customer
    WHERE salary = (
        SELECT DISTINCT salary
        FROM customer
        ORDER BY salary DESC
        LIMIT 1 OFFSET 1   -- 跳 1 个唯一值取下一个 / skip 1 distinct value
    );
""").df()


In [ ]:
# 解法 C: correlated 自连接 (经典但慢)
# Classic correlated self-join — O(n²)
conn.sql("""
    SELECT name, salary FROM customer c1
    WHERE 1 = (
        SELECT COUNT(DISTINCT salary) FROM customer c2
        WHERE c2.salary > c1.salary
    );
""").df()


**三种都返回**: Alice (95000) + Diana (95000)（并列第 2 高）。
All three return both ties.

> 💡 **面试推荐写法 = `DENSE_RANK`**：清晰、`O(n log n)`、自动处理并列。**避开** `RANK()`（会跳号）和`ROW_NUMBER()`（在并列时会"挑一个"）。
> Use `DENSE_RANK`: clear, efficient, handles ties correctly. Avoid `RANK` (skips numbers) and `ROW_NUMBER` (arbitrary pick on tie).


<a id="2"></a>
## Problem 2 · 中位数（手算）/ Median Without MEDIAN

> "在**不用 `MEDIAN` / `PERCENTILE_CONT` 函数**的前提下，算出工资中位数。"

DuckDB / Postgres 有现成函数，但 **MySQL 没有**——这就是面试题的来源。
DuckDB / Postgres have a function; MySQL doesn't — that's why this is asked.

### 思路 / Approach

中位数定义：排序后**中间位置**的值。
- 行数为奇数 $n$ → 第 $(n+1)/2$ 个值
- 行数为偶数 $n$ → 第 $n/2$ 和 $(n/2)+1$ 个值的平均

**窗口函数解法**：每行算自己的 `ROW_NUMBER` 和总行数，挑"位置在中间一两行"的。


In [ ]:
conn.sql("""
    WITH ranked AS (
        SELECT
            salary,
            ROW_NUMBER() OVER (ORDER BY salary)  AS rn,
            COUNT(*)     OVER ()                  AS n
        FROM customer
    )
    SELECT
        AVG(salary) AS median
    FROM ranked
    -- 偶数和奇数都覆盖：取最中间的 1 或 2 行
    WHERE rn IN ((n + 1) / 2, (n + 2) / 2);
""").df()


**公式 `(n+1)/2` 和 `(n+2)/2`** 巧妙地：
- 当 $n$ 奇数 → 两者相同 → 取 1 行
- 当 $n$ 偶数 → 一前一后 → 取 2 行 → `AVG` 出中位数

The formula `(n+1)/2, (n+2)/2`:
- odd $n$ → both equal → 1 row
- even $n$ → two adjacent rows → AVG gives the median.

我们有 8 个客户（偶数），中位数 = 第 4 高和第 5 高的平均。


<a id="3"></a>
## Problem 3 · 买过"全部"商品（关系除法）⭐ / Relational Division

> "**找出买过所有 product_required 商品**的用户。"
> "Find users who have bought EVERY product in `product_required`."

这叫**关系除法**——SQL 没有原生 `÷`，要手写。有 3 种经典写法：
"Relational division" — SQL has no native `÷`. Three classic patterns:

### 思路 1 / Approach 1: COUNT DISTINCT 比对

"该用户买过的**不同必买商品数** = 必买总数" → 即买全。


In [ ]:
conn.sql("""
    SELECT o.user_id
    FROM orders_demo AS o
    JOIN product_required AS p USING (product_id)
    GROUP BY o.user_id
    HAVING COUNT(DISTINCT o.product_id) = (SELECT COUNT(*) FROM product_required);
""").df()


### 思路 2 / Approach 2: 双重 NOT EXISTS（教科书写法）

"**不存在** 必买商品 $p$，使得**不存在** 该用户买过 $p$"——这就是关系代数除法的字面翻译。
"There's no required product $p$ such that no order of this user covers $p$."


In [ ]:
conn.sql("""
    SELECT DISTINCT o.user_id
    FROM orders_demo AS o
    WHERE NOT EXISTS (
        SELECT 1 FROM product_required AS p
        WHERE NOT EXISTS (
            SELECT 1 FROM orders_demo AS o2
            WHERE o2.user_id = o.user_id
              AND o2.product_id = p.product_id
        )
    );
""").df()


**两种都返回**: user 1, user 3（买全 3 个）。User 2 只买了 2 个，user 5 只买了 1 个，都不出现。
Both return users 1 and 3.

**思路 1 更直观、更快**；思路 2 是关系代数的"字面翻译"，理解关系除法语义更深。
Approach 1 is more intuitive and faster; approach 2 is the textbook relational-algebra form.


<a id="4"></a>
## Problem 4 · Pivot 行转列 / Rows-to-columns

> "把'每用户、每事件类型的次数' 从**长表**变成 **宽表**（每事件类型一列）。"
> "Reshape long → wide: one column per event_type."

### 思路：条件聚合 + GROUP BY


In [ ]:
# 原始长表 / Source long table
conn.sql("""
    SELECT user_id, event_type, COUNT(*) AS n
    FROM events
    GROUP BY user_id, event_type
    ORDER BY user_id, event_type;
""").df()


In [ ]:
# Pivot: 每事件类型一列 / Pivot to wide format
conn.sql("""
    SELECT
        user_id,
        COUNT(*) FILTER (WHERE event_type = 'view') AS views,
        COUNT(*) FILTER (WHERE event_type = 'play') AS plays,
        COUNT(*) FILTER (WHERE event_type = 'like') AS likes,
        COUNT(*) FILTER (WHERE event_type = 'buy')  AS buys,
        COUNT(*)                                     AS total
    FROM events
    GROUP BY user_id
    ORDER BY user_id;
""").df()


**`COUNT(*) FILTER (WHERE ...)`** 是现代写法。MySQL 等价：`SUM(CASE WHEN event_type='view' THEN 1 ELSE 0 END)`。
`COUNT(*) FILTER` is the modern form; MySQL needs the SUM-CASE-WHEN trick.

> 💡 DuckDB 还有 **`PIVOT` 关键字**（同 Snowflake / BigQuery），动态展开值时更方便：
> DuckDB also supports the `PIVOT` keyword for dynamic columns:
> ```sql
> SELECT * FROM (SELECT user_id, event_type FROM events)
> PIVOT (COUNT(*) FOR event_type IN ('view', 'play', 'like', 'buy'));
> ```


<a id="5"></a>
## Problem 5 · 转化漏斗 / Funnel ⭐

> "**4 步漏斗**：每一步多少人完成？转化率多少？"
> "4-step funnel: count and conversion at each stage."

业务漏斗：`view → play → like → buy`。**每一步比上一步**少多少？
View → Play → Like → Buy. Drop-off at each step?

### 思路：每用户每事件至少有过 → 条件聚合


In [ ]:
conn.sql("""
    WITH per_user AS (
        SELECT
            user_id,
            BOOL_OR(event_type = 'view') AS did_view,
            BOOL_OR(event_type = 'play') AS did_play,
            BOOL_OR(event_type = 'like') AS did_like,
            BOOL_OR(event_type = 'buy')  AS did_buy
        FROM events
        GROUP BY user_id
    )
    SELECT
        COUNT(*) FILTER (WHERE did_view) AS s1_view,
        COUNT(*) FILTER (WHERE did_play) AS s2_play,
        COUNT(*) FILTER (WHERE did_like) AS s3_like,
        COUNT(*) FILTER (WHERE did_buy)  AS s4_buy,
        ROUND(100.0 * COUNT(*) FILTER (WHERE did_play)
                / NULLIF(COUNT(*) FILTER (WHERE did_view), 0), 1) AS view_to_play_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE did_buy)
                / NULLIF(COUNT(*) FILTER (WHERE did_view), 0), 1) AS overall_conv_pct
    FROM per_user;
""").df()


**严格漏斗**（必须**按顺序**才算）写法更复杂——需要每用户每事件**第一次**的时间戳，然后判断 `play_ts > view_ts AND like_ts > play_ts ...`：
For a **strict ordered** funnel (each step must come after the previous), you need first-event timestamps and chained inequalities:

```sql
WITH first_ts AS (
    SELECT
        user_id,
        MIN(ts) FILTER (WHERE event_type='view') AS view_ts,
        MIN(ts) FILTER (WHERE event_type='play') AS play_ts,
        MIN(ts) FILTER (WHERE event_type='like') AS like_ts,
        MIN(ts) FILTER (WHERE event_type='buy')  AS buy_ts
    FROM events
    GROUP BY user_id
)
SELECT
    COUNT(*) FILTER (WHERE view_ts IS NOT NULL) AS s1,
    COUNT(*) FILTER (WHERE play_ts > view_ts) AS s2,
    COUNT(*) FILTER (WHERE like_ts > play_ts) AS s3,
    COUNT(*) FILTER (WHERE buy_ts  > like_ts) AS s4
FROM first_ts;
```


<a id="6"></a>
## Problem 6 · Sessionization ⭐

> "把同一用户的连续事件切成 session：**与上一个事件间隔 > 30 分钟 = 新 session**。"
> "Split consecutive events into sessions: gap > 30 min = new session."

**面试 ★★★★★**——Uber / Meta / Google 都问过。
Asked at Uber / Meta / Google.

### 思路 / Approach

1. 用 `LAG` 拿到每行的上一个事件时间
2. 标记"是不是新 session 起点"（间隔 > 30 min 或者第一行）
3. **累计 `SUM`** 那个标记 → 得到 `session_id` ⭐


In [ ]:
conn.sql("""
    WITH with_gap AS (
        SELECT
            event_id,
            user_id,
            event_type,
            ts,
            LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) AS prev_ts,
            CASE
                WHEN LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) IS NULL
                  OR ts - LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) > INTERVAL 30 MINUTE
                THEN 1 ELSE 0
            END AS is_new_session
        FROM events
    ),
    sessions AS (
        SELECT
            event_id, user_id, event_type, ts,
            SUM(is_new_session) OVER (
                PARTITION BY user_id ORDER BY ts
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS session_num
        FROM with_gap
    )
    SELECT
        user_id,
        session_num,
        MIN(ts)         AS session_start,
        MAX(ts)         AS session_end,
        COUNT(*)        AS n_events,
        STRING_AGG(event_type, ' → ' ORDER BY ts) AS event_sequence
    FROM sessions
    GROUP BY user_id, session_num
    ORDER BY user_id, session_num;
""").df()


**看 user 2**：被切成 2 个 session —— 上午 11:00 一个，下午 14:30 另一个（间隔 > 30 min）。
User 2 is split into 2 sessions because the 11:05→14:30 gap exceeds 30 min.

> 💡 **`SUM(0/1 flag) OVER (ORDER BY)`** 是把"二值标记"变成"序列 ID"的**核心套路**。**牢记**。
> The `SUM(0/1 flag) OVER (ORDER BY)` idiom converts a "new island" flag into a running ID. **The most-reused trick in interview SQL.**


<a id="7"></a>
## Problem 7 · 留存率 D1 / D7 / D30 ⭐

> "**D1 留存** = 注册后第 1 天回来的比例。D7 / D30 类似。"
> "D1 retention = % of users who returned 1 day after signup. D7 / D30 similar."

### 思路 / Approach

1. 每用户的"首次活跃日期" = signup
2. 用 self-JOIN（或 anti-join）判断"signup + N 天"那天是否有事件


In [ ]:
conn.sql("""
    WITH signup AS (
        SELECT
            user_id,
            MIN(CAST(ts AS DATE)) AS signup_date
        FROM events
        GROUP BY user_id
    ),
    activity AS (
        SELECT DISTINCT user_id, CAST(ts AS DATE) AS active_date
        FROM events
    )
    SELECT
        COUNT(*)                                                           AS n_users,
        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1 FROM activity a
                WHERE a.user_id = s.user_id
                  AND a.active_date = s.signup_date + INTERVAL 1 DAY
            )
        ) AS retained_d1,
        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1 FROM activity a
                WHERE a.user_id = s.user_id
                  AND a.active_date = s.signup_date + INTERVAL 3 DAY
            )
        ) AS retained_d3,
        ROUND(100.0 * COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1 FROM activity a
                WHERE a.user_id = s.user_id
                  AND a.active_date = s.signup_date + INTERVAL 1 DAY
            )
        ) / NULLIF(COUNT(*), 0), 1) AS d1_rate_pct
    FROM signup AS s;
""").df()


**5 个用户里有 X 个在 signup+1 天回来 = D1 留存 X/5**。Production 系统里同一模板算 D7 / D30 只需改 INTERVAL。
Same template; just change INTERVAL for D7/D30.


<a id="8"></a>
## Problem 8 · Cohort 分析

> "**按 signup 月份**分 cohort，看每个 cohort 在后续 N 个月的留存率。"
> "Group users by signup month; track retention in subsequent months."

经典"**Cohort table**"——大数据分析师必备。
The classic cohort retention table.


In [ ]:
conn.sql("""
    WITH signup AS (
        SELECT
            user_id,
            DATE_TRUNC('month', MIN(ts)) AS cohort_month
        FROM events
        GROUP BY user_id
    ),
    activity AS (
        SELECT DISTINCT user_id, DATE_TRUNC('month', ts) AS active_month
        FROM events
    ),
    cohort_activity AS (
        SELECT
            s.cohort_month,
            DATEDIFF('month', s.cohort_month, a.active_month) AS month_offset,
            a.user_id
        FROM signup AS s
        JOIN activity AS a USING (user_id)
        WHERE a.active_month >= s.cohort_month
    )
    SELECT
        cohort_month,
        month_offset,
        COUNT(DISTINCT user_id) AS active_users
    FROM cohort_activity
    GROUP BY cohort_month, month_offset
    ORDER BY cohort_month, month_offset;
""").df()


**所有事件都在 6 月**，所以只有 `2026-06-01` cohort + `month_offset = 0`。真实生产数据 cohort 表会是个对角阵。
All events are in June so we see only the June cohort at offset 0. Real production data would yield a triangular cohort matrix.


<a id="9"></a>
## Problem 9 · 找/删重复 / Find & Dedup Duplicates

### 9a) 找重复 / Find duplicate (e.g. same email)

> "找出重复的薪资记录。"


In [ ]:
conn.sql("""
    SELECT salary, COUNT(*) AS n
    FROM customer
    GROUP BY salary
    HAVING COUNT(*) > 1
    ORDER BY n DESC;
""").df()


### 9b) 标记"组内是第几条重复" + 只保留一条 / Mark duplicates, keep one

经典套路：`ROW_NUMBER() OVER (PARTITION BY dup_key ORDER BY tiebreaker)`，然后 `WHERE rn = 1`。


In [ ]:
conn.sql("""
    WITH ranked AS (
        SELECT
            customer_id, name, salary,
            ROW_NUMBER() OVER (PARTITION BY salary ORDER BY customer_id) AS rn
        FROM customer
    )
    SELECT * FROM ranked
    WHERE rn = 1                  -- 每组保留 ID 最小的那一条 / Keep first per group
    ORDER BY salary DESC;
""").df()


真实业务里 dedup 模板大半是 `ROW_NUMBER + WHERE rn = 1` —— **牢记**。
The `ROW_NUMBER + WHERE rn = 1` template is the universal dedup pattern.


<a id="10"></a>
## Problem 10 · 经理层级 / Manager Chain ⭐

> "把员工的**所有上级**列出来（CEO 是顶层）。"
> "List the full management chain for each employee, up to the CEO."

经典 **recursive CTE** 题——LinkedIn / Meta 都问过。
Classic recursive CTE — asked at LinkedIn / Meta.


In [ ]:
conn.sql("""
    WITH RECURSIVE chain AS (
        -- anchor: 每个员工的"第一层 = 自己"
        SELECT
            emp_id      AS root_emp,
            emp_id,
            name,
            manager_id,
            0 AS depth
        FROM employee

        UNION ALL

        -- recursive: 上一层的 manager 是这一层的 emp
        SELECT
            chain.root_emp,
            e.emp_id,
            e.name,
            e.manager_id,
            chain.depth + 1
        FROM chain
        JOIN employee AS e
          ON chain.manager_id = e.emp_id
    )
    SELECT
        root_emp AS for_employee,
        depth,
        name     AS in_chain
    FROM chain
    WHERE root_emp IN (7, 10)        -- 只看 Eng Erik 和 Eng Eva / show 2 employees
    ORDER BY root_emp, depth;
""").df()


**看 Eng Erik (emp_id=7)** 的链：
- depth 0 = Erik 自己
- depth 1 = Mgr Mark（他的 manager）
- depth 2 = VP Vincent
- depth 3 = CEO Carla（顶层）

完整管理链一次性查出。**面试时秒答这道题就能拿分**。
Knowing the recursive-CTE manager chain pattern earns easy interview points.

### 经典变种 / Classic variants

- "每个员工的**直接经理**" → JOIN once (no recursion)
- "每个员工的**所有下属**（含间接）" → recursion on the other direction
- "找**没有下属**的员工" → anti-join: `WHERE NOT EXISTS (SELECT 1 FROM employee e2 WHERE e2.manager_id = e.emp_id)`


<a id="11"></a>
## 11. 小结 / Summary

### 10 大模板速查 / 10 templates cheat sheet

| 题型 | 模板 |
|---|---|
| **N-th 高** | `DENSE_RANK() OVER (ORDER BY ...) WHERE rk = N` |
| **中位数** | `WHERE rn IN ((n+1)/2, (n+2)/2)` + `AVG` |
| **买过全部** | `HAVING COUNT(DISTINCT ...) = (SELECT COUNT(*) FROM required)` |
| **Pivot 行转列** | `COUNT(*) FILTER (WHERE type = 'X')` |
| **漏斗** | 每用户 `BOOL_OR(event = 'view')` + 多步 FILTER |
| **Sessionization** ⭐ | `SUM(is_new_flag) OVER (PARTITION BY user ORDER BY ts)` |
| **留存率** | `signup + COUNT FILTER WHERE EXISTS (signup + N day)` |
| **Cohort** | 双 GROUP BY: cohort_month × month_offset |
| **Dedup** | `ROW_NUMBER OVER (PARTITION BY key) WHERE rn = 1` |
| **管理链** | `WITH RECURSIVE` + UNION ALL + JOIN on manager_id |

### 💡 面试速查 / Interview must-knows

1. **N-th 高**：永远用 `DENSE_RANK`，不要用 `ROW_NUMBER` 或 `RANK`
2. **没有 MEDIAN 函数**：`(n+1)/2, (n+2)/2` 一次盖住奇偶
3. **"买过全部"**：`COUNT DISTINCT = required count` 是最快写法
4. **`COUNT(*) FILTER (WHERE ...)`** = pivot 神技 + 漏斗神技
5. **`SUM(flag) OVER (ORDER BY)`** = sessionization 神技
6. **Dedup**：`ROW_NUMBER WHERE rn = 1` 是工业标准
7. **管理链** = recursive CTE 必背
8. **每个模板**都要会"中文一句话说清思路"——面试官会让你解释

### 下一节预告 / Next up

**Part 1.7 · 索引与执行计划** —— `EXPLAIN`、B-tree 索引、何时不用索引、查询优化器在干啥。
**Part 1.7 · Indexes & EXPLAIN** — query plans, B-tree indexes, when indexes don't help.
